# Content-Based Baseline Recommender

## Project
An Explainable Knowledge Graph-Driven Hybrid Recommender for Intelligent Task Allocation in Distributed Onboarding Operations

## Purpose

This notebook develops the first recommendation baseline for the intelligent task allocation system.

The content-based recommender compares onboarding task requirements with agent profiles using information such as:

- Required skills
- Agent skills and proficiency
- Agent specialization
- Task and agent location
- Availability
- Workload
- Performance score
- Experience

The purpose of this baseline is to determine whether suitable onboarding agents can be ranked for a given task before collaborative filtering and knowledge graph components are introduced.

The current experiment uses synthetic development data while awaiting the approved industry dataset. Results from this dataset are for development and pipeline verification only and should not be interpreted as final real-world performance.

## Experiment Information

- Run ID: CBF_RUN_001
- Model: Content-Based Baseline
- Dataset Version: Synthetic Development Dataset v1
- Random Seed: 42
- Training Split: 70%
- Validation Split: 15%
- Test Split: 15%
- Development data: Synthetic
- Test data will remain separate during model development

## 1. Import Required Libraries

The baseline recommender uses pandas and NumPy for data manipulation and numerical operations.

The prepared datasets from the previous notebook are loaded from the project data folders.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

## 2. Set Experiment Configuration

A fixed random seed is maintained for reproducibility.

The notebook also identifies the project folders containing the raw and processed development data.

In [2]:
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

project_root = Path.cwd().parent

raw_data_path = project_root / "data" / "raw"
processed_data_path = project_root / "data" / "processed"

print("Project root:", project_root)
print("Raw data:", raw_data_path)
print("Processed data:", processed_data_path)

Project root: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation
Raw data: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\data\raw
Processed data: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\data\processed


## 3. Load Prepared Development Data

The datasets generated during the data preparation stage are loaded for baseline recommender development.

The raw datasets provide agent, skill and task profile information, while the processed datasets contain the separated training, validation and test interactions.

In [3]:
agents_df = pd.read_csv(
    raw_data_path / "agents.csv"
)

skills_df = pd.read_csv(
    raw_data_path / "skills.csv"
)

agent_skills_df = pd.read_csv(
    raw_data_path / "agent_skills.csv"
)

tasks_df = pd.read_csv(
    raw_data_path / "tasks.csv"
)

task_skills_df = pd.read_csv(
    raw_data_path / "task_skills.csv"
)

historical_assignments_df = pd.read_csv(
    raw_data_path / "historical_assignments.csv"
)

train_df = pd.read_csv(
    processed_data_path / "train_interactions.csv"
)

validation_df = pd.read_csv(
    processed_data_path / "validation_interactions.csv"
)

test_df = pd.read_csv(
    processed_data_path / "test_interactions.csv"
)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [4]:
dataset_summary = pd.DataFrame({
    "Dataset": [
        "Agents",
        "Skills",
        "Agent Skills",
        "Tasks",
        "Task Skills",
        "Historical Assignments",
        "Training Interactions",
        "Validation Interactions",
        "Test Interactions"
    ],

    "Records": [
        len(agents_df),
        len(skills_df),
        len(agent_skills_df),
        len(tasks_df),
        len(task_skills_df),
        len(historical_assignments_df),
        len(train_df),
        len(validation_df),
        len(test_df)
    ]
})

dataset_summary

,Dataset,Records
0,Agents,100
1,Skills,8
2,Agent Skills,298
3,Tasks,2000
4,Task Skills,3887
5,Historical Assignments,1607
6,Training Interactions,912
7,Validation Interactions,196
8,Test Interactions,196


### 3.1 Verify Train, Validation and Test Separation

Before developing the baseline, the interaction identifiers are checked again to confirm that the training, validation and test sets remain independent.

The test set will not be used during baseline development or parameter adjustment.

In [5]:
train_ids = set(train_df["assignment_id"])
validation_ids = set(validation_df["assignment_id"])
test_ids = set(test_df["assignment_id"])

print(
    "Train / Validation overlap:",
    len(train_ids & validation_ids)
)

print(
    "Train / Test overlap:",
    len(train_ids & test_ids)
)

print(
    "Validation / Test overlap:",
    len(validation_ids & test_ids)
)

Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0


### Interpretation

All required development datasets were loaded successfully.

The training, validation and test interaction datasets remain independent, with no overlapping assignment records.

The test dataset will remain untouched during baseline development. The validation dataset will be used to assess the initial content-based recommender before later comparison with collaborative filtering, knowledge graph and hybrid approaches.

## 4. Prepare Agent and Task Feature Profiles

The content-based baseline compares the requirements of an onboarding task with the characteristics of available agents.

The baseline uses:

- Skill compatibility
- Specialization match
- Location match
- Availability
- Workload
- Performance score
- Experience

Skill proficiency and task-required proficiency are converted into numerical levels so that they can be compared consistently.

The feature preparation performed in this section is deterministic and does not use the validation or test outcomes.

In [7]:
proficiency_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

required_level_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

### 4.1 Create Agent Skill Profiles

Each agent is represented by a dictionary containing the skills they possess and their proficiency levels.

In [8]:
agent_skill_profiles = {}

for agent_id, group in agent_skills_df.groupby("agent_id"):

    agent_skill_profiles[agent_id] = {
        row["skill_id"]:
            proficiency_map[row["proficiency_level"]]
        for _, row in group.iterrows()
    }


print(
    "Number of agent skill profiles:",
    len(agent_skill_profiles)
)

print(
    "\nExample agent profile:"
)

example_agent_id = agents_df.iloc[0]["agent_id"]

print(
    example_agent_id,
    agent_skill_profiles[example_agent_id]
)

Number of agent skill profiles: 100

Example agent profile:
A001 {'S001': 2, 'S006': 3}


### 4.2 Create Task Skill Profiles

Each task is represented by the skills it requires and the minimum proficiency level expected for each skill.

In [9]:
task_skill_profiles = {}

for task_id, group in task_skills_df.groupby("task_id"):

    task_skill_profiles[task_id] = {
        row["skill_id"]:
            required_level_map[
                row["required_level"]
            ]
        for _, row in group.iterrows()
    }


print(
    "Number of task skill profiles:",
    len(task_skill_profiles)
)

example_task_id = tasks_df.iloc[0]["task_id"]

print(
    "\nExample task profile:"
)

print(
    example_task_id,
    task_skill_profiles[example_task_id]
)

Number of task skill profiles: 2000

Example task profile:
T001 {'S003': 2}


### 4.3 Create Agent and Task Lookup Tables

Agent and task records are indexed by their identifiers so that the recommender can quickly retrieve the information required during scoring.

In [10]:
agents_lookup = (
    agents_df
    .set_index("agent_id")
    .to_dict("index")
)

tasks_lookup = (
    tasks_df
    .set_index("task_id")
    .to_dict("index")
)

print(
    "Agent lookup records:",
    len(agents_lookup)
)

print(
    "Task lookup records:",
    len(tasks_lookup)
)

Agent lookup records: 100
Task lookup records: 2000


### 4.4 Normalize Numerical Agent Features

Performance, workload and experience are measured on different scales.

They are converted into values between 0 and 1 so that they can be combined fairly when calculating recommendation scores.

A lower workload is treated as more desirable for task allocation.

In [11]:
max_workload = agents_df["workload"].max()
max_experience = agents_df["experience_years"].max()

agents_features_df = agents_df.copy()

agents_features_df["performance_normalized"] = (
    agents_features_df["performance_score"] / 100
)

agents_features_df["workload_score"] = 1 - (
    agents_features_df["workload"] / max_workload
)

agents_features_df["experience_normalized"] = (
    agents_features_df["experience_years"] / max_experience
)

agents_features_df[
    [
        "agent_id",
        "performance_score",
        "performance_normalized",
        "workload",
        "workload_score",
        "experience_years",
        "experience_normalized"
    ]
].head(10)

,agent_id,performance_score,performance_normalized,workload,workload_score,experience_years,experience_normalized
0,A001,68.93,0.6893,1,0.8,1,0.125
1,A002,63.75,0.6375,0,1.0,3,0.375
2,A003,88.05,0.8805,5,0.0,6,0.750
3,A004,90.35,0.9035,0,1.0,2,0.250
4,A005,98.29,0.9829,1,0.8,5,0.625
5,A006,93.90,0.9390,2,0.6,4,0.500
6,A007,98.92,0.9892,0,1.0,6,0.750
7,A008,84.74,0.8474,5,0.0,5,0.625
8,A009,69.12,0.6912,5,0.0,4,0.500
9,A010,71.12,0.7112,3,0.4,5,0.625


In [12]:
print(
    "Performance range:",
    agents_features_df["performance_normalized"].min(),
    "to",
    agents_features_df["performance_normalized"].max()
)

print(
    "Workload score range:",
    agents_features_df["workload_score"].min(),
    "to",
    agents_features_df["workload_score"].max()
)

print(
    "Experience range:",
    agents_features_df["experience_normalized"].min(),
    "to",
    agents_features_df["experience_normalized"].max()
)

Performance range: 0.6084 to 0.9997
Workload score range: 0.0 to 1.0
Experience range: 0.0 to 1.0


## 5. Develop the Content-Based Suitability Score

The baseline recommender calculates how suitable each onboarding agent is for a particular task.

Unavailable agents are excluded from recommendation.

For available agents, the suitability score combines:

- Skill compatibility: 45%
- Specialization match: 20%
- Location match: 10%
- Performance: 10%
- Workload: 10%
- Experience: 5%

Skill compatibility receives the highest weight because possessing the capabilities required by a task is the most important requirement for allocation.

These weights represent an initial baseline configuration. They are not treated as final optimal values and may later be adjusted using validation evidence.

In [13]:
BASELINE_WEIGHTS = {
    "skill_match": 0.45,
    "specialization_match": 0.20,
    "location_match": 0.10,
    "performance": 0.10,
    "workload": 0.10,
    "experience": 0.05
}

print(
    "Total weight:",
    sum(BASELINE_WEIGHTS.values())
)

Total weight: 1.0


### 5.1 Create Agent Feature Lookup

The normalized agent features are indexed by agent ID so that they can be retrieved efficiently during recommendation scoring.

In [14]:
agent_feature_lookup = (
    agents_features_df
    .set_index("agent_id")
    .to_dict("index")
)

print(
    "Agent feature records:",
    len(agent_feature_lookup)
)

Agent feature records: 100


### 5.2 Calculate Skill Compatibility

Skill compatibility compares the proficiency level of an agent with the proficiency required by a task.

A score of 1 means the agent fully satisfies the required skill level.

A partial score is given where the agent possesses the skill but has a lower proficiency level.

A missing required skill receives a score of 0.

In [15]:
def calculate_skill_match(
    agent_id,
    task_id
):
    agent_skills = agent_skill_profiles.get(
        agent_id,
        {}
    )

    required_skills = task_skill_profiles.get(
        task_id,
        {}
    )

    if not required_skills:
        return 0.0

    skill_scores = []

    for skill_id, required_level in required_skills.items():

        agent_level = agent_skills.get(
            skill_id,
            0
        )

        if agent_level == 0:
            score = 0.0

        else:
            score = min(
                agent_level / required_level,
                1.0
            )

        skill_scores.append(score)

    return float(
        np.mean(skill_scores)
    )

In [16]:
print(
    "Example skill match:",
    calculate_skill_match(
        example_agent_id,
        example_task_id
    )
)

Example skill match: 0.0


### 5.3 Calculate Overall Agent Suitability

The overall content-based suitability score combines skill compatibility with specialization, location, performance, workload and experience.

Unavailable agents are excluded before ranking.

The function also returns the individual score components so that the recommendation can later be explained to the supervisor.

In [17]:
def calculate_agent_suitability(
    agent_id,
    task_id
):

    agent = agent_feature_lookup[
        agent_id
    ]

    task = tasks_lookup[
        task_id
    ]

    # Exclude unavailable agents
    if not bool(agent["availability"]):
        return None

    skill_match = calculate_skill_match(
        agent_id,
        task_id
    )

    specialization_match = (
        1.0
        if agent["specialization"]
        == task["task_type"]
        else 0.0
    )

    location_match = (
        1.0
        if agent["location"]
        == task["location"]
        else 0.0
    )

    performance_score = agent[
        "performance_normalized"
    ]

    workload_score = agent[
        "workload_score"
    ]

    experience_score = agent[
        "experience_normalized"
    ]

    total_score = (
        BASELINE_WEIGHTS["skill_match"]
        * skill_match

        + BASELINE_WEIGHTS[
            "specialization_match"
        ]
        * specialization_match

        + BASELINE_WEIGHTS[
            "location_match"
        ]
        * location_match

        + BASELINE_WEIGHTS["performance"]
        * performance_score

        + BASELINE_WEIGHTS["workload"]
        * workload_score

        + BASELINE_WEIGHTS["experience"]
        * experience_score
    )

    return {
        "agent_id": agent_id,

        "score": round(
            float(total_score),
            4
        ),

        "skill_match": round(
            skill_match,
            4
        ),

        "specialization_match":
            specialization_match,

        "location_match":
            location_match,

        "performance_score": round(
            float(performance_score),
            4
        ),

        "workload_score": round(
            float(workload_score),
            4
        ),

        "experience_score": round(
            float(experience_score),
            4
        )
    }

### 5.4 Rank Suitable Agents

The recommendation function calculates suitability for all available agents and ranks them from highest to lowest score.

The function returns the top agents for a selected onboarding task.

In [18]:
def recommend_agents(
    task_id,
    top_k=5
):

    recommendations = []

    for agent_id in agents_features_df[
        "agent_id"
    ]:

        result = calculate_agent_suitability(
            agent_id,
            task_id
        )

        if result is not None:
            recommendations.append(
                result
            )

    recommendations_df = pd.DataFrame(
        recommendations
    )

    recommendations_df = (
        recommendations_df
        .sort_values(
            "score",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    recommendations_df.insert(
        0,
        "rank",
        range(
            1,
            len(recommendations_df) + 1
        )
    )

    return recommendations_df

In [20]:
pending_tasks = tasks_df[
    tasks_df["status"] == "Pending"
]

sample_task_id = (
    pending_tasks.iloc[0][
        "task_id"
    ]
)

sample_task = tasks_lookup[
    sample_task_id
]

print(
    "Task ID:",
    sample_task_id
)

print(
    "Task Type:",
    sample_task["task_type"]
)

print(
    "Location:",
    sample_task["location"]
)

print(
    "Priority:",
    sample_task["priority"]
)

print(
    "Difficulty:",
    sample_task["difficulty"]
)

Task ID: T004
Task Type: KYC Verification
Location: Kisumu
Priority: Medium
Difficulty: Low


In [21]:
top_recommendations = (
    recommend_agents(
        sample_task_id,
        top_k=5
    )
)

top_recommendations

,rank,agent_id,score,skill_match,specialization_match,location_match,performance_score,workload_score,experience_score
0,1,A043,0.8834,1.00,1.0,0.0,0.8337,1.0,1.000
1,2,A040,0.8044,1.00,1.0,0.0,0.7061,0.4,0.875
2,3,A017,0.7166,1.00,0.0,1.0,0.8038,0.8,0.125
3,4,A059,0.6822,0.50,1.0,1.0,0.7719,0.8,0.000
4,5,A080,0.6682,0.75,1.0,0.0,0.8445,0.4,0.125


### Interpretation

The content-based baseline successfully produced a ranked list of available onboarding agents for a selected task.

The ranking combines task-agent skill compatibility with specialization, location, workload, performance and experience.

The individual score components are retained so that the recommendation can be explained rather than presented as an unexplained numerical ranking.

The current weighting scheme represents the initial baseline configuration and will be assessed using validation data before comparison with more advanced recommendation approaches.

## 6. Generate Recommendation Explanations

The project requires recommendations to be explainable.

Instead of displaying only a numerical suitability score, the system generates a short explanation describing the main factors that contributed to an agent's ranking.

The explanation considers skill compatibility, specialization, location, workload, performance and experience.

In [22]:
def generate_explanation(
    agent_id,
    task_id,
    scoring_result
):

    agent = agent_feature_lookup[agent_id]
    task = tasks_lookup[task_id]

    reasons = []

    # Skill explanation
    if scoring_result["skill_match"] == 1.0:
        reasons.append(
            "fully matches the required skills"
        )

    elif scoring_result["skill_match"] >= 0.5:
        reasons.append(
            "partially matches the required skills"
        )

    else:
        reasons.append(
            "has limited matching skills"
        )

    # Specialization
    if scoring_result[
        "specialization_match"
    ] == 1.0:
        reasons.append(
            "has the required task specialization"
        )

    # Location
    if scoring_result[
        "location_match"
    ] == 1.0:
        reasons.append(
            "is located in the same area as the task"
        )

    # Workload
    if scoring_result[
        "workload_score"
    ] >= 0.8:
        reasons.append(
            "currently has a low workload"
        )

    # Performance
    if scoring_result[
        "performance_score"
    ] >= 0.8:
        reasons.append(
            "has strong historical performance"
        )

    # Experience
    if scoring_result[
        "experience_score"
    ] >= 0.75:
        reasons.append(
            "has strong relevant experience"
        )

    return "; ".join(reasons) + "."

In [23]:
explained_recommendations = (
    top_recommendations.copy()
)

explained_recommendations[
    "explanation"
] = explained_recommendations.apply(
    lambda row: generate_explanation(
        row["agent_id"],
        sample_task_id,
        row
    ),
    axis=1
)

explained_recommendations[
    [
        "rank",
        "agent_id",
        "score",
        "explanation"
    ]
]

,rank,agent_id,score,explanation
0,1,A043,0.8834,fully matches the required skills; has the req...
1,2,A040,0.8044,fully matches the required skills; has the req...
2,3,A017,0.7166,fully matches the required skills; is located ...
3,4,A059,0.6822,partially matches the required skills; has the...
4,5,A080,0.6682,partially matches the required skills; has the...


### Interpretation

The baseline recommender now provides both a ranked suitability score and a human-readable explanation for each recommended onboarding agent.

For example, a highly ranked agent may be recommended because they fully match the required skills, have the correct specialization, maintain a low workload and demonstrate strong historical performance.

This improves transparency by allowing the supervisor to understand why an agent was recommended instead of relying only on an unexplained ranking score.

The explanations generated at this stage are rule-based explanations derived directly from the content-based score components.

## 7. Baseline Validation Evaluation

The content-based baseline is evaluated using the validation interaction dataset.

For each historical task, the agent who actually handled the task is treated as the relevant agent.

The recommender then ranks candidate agents for the same task and checks whether the historical agent appears within the Top-K recommendations.

Current agent availability is not enforced during historical ranking evaluation because the availability field represents the current agent profile rather than availability at the historical assignment time.

The operational recommender will continue to enforce availability when recommending agents for new tasks.

In [24]:
def recommend_agents(
    task_id,
    top_k=5,
    enforce_availability=True
):

    recommendations = []

    for agent_id in agents_features_df["agent_id"]:

        result = calculate_agent_suitability(
            agent_id,
            task_id
        )

        # For operational recommendations,
        # unavailable agents remain excluded
        if result is None:

            if enforce_availability:
                continue

            # Historical evaluation:
            # score agent without enforcing current availability
            agent = agent_feature_lookup[agent_id]
            task = tasks_lookup[task_id]

            skill_match = calculate_skill_match(
                agent_id,
                task_id
            )

            specialization_match = (
                1.0
                if agent["specialization"]
                == task["task_type"]
                else 0.0
            )

            location_match = (
                1.0
                if agent["location"]
                == task["location"]
                else 0.0
            )

            total_score = (
                BASELINE_WEIGHTS["skill_match"]
                * skill_match

                + BASELINE_WEIGHTS["specialization_match"]
                * specialization_match

                + BASELINE_WEIGHTS["location_match"]
                * location_match

                + BASELINE_WEIGHTS["performance"]
                * agent["performance_normalized"]

                + BASELINE_WEIGHTS["workload"]
                * agent["workload_score"]

                + BASELINE_WEIGHTS["experience"]
                * agent["experience_normalized"]
            )

            result = {
                "agent_id": agent_id,
                "score": round(float(total_score), 4),
                "skill_match": round(skill_match, 4),
                "specialization_match": specialization_match,
                "location_match": location_match,
                "performance_score": round(
                    float(agent["performance_normalized"]),
                    4
                ),
                "workload_score": round(
                    float(agent["workload_score"]),
                    4
                ),
                "experience_score": round(
                    float(agent["experience_normalized"]),
                    4
                )
            }

        recommendations.append(result)

    recommendations_df = pd.DataFrame(
        recommendations
    )

    recommendations_df = (
        recommendations_df
        .sort_values(
            "score",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    recommendations_df.insert(
        0,
        "rank",
        range(1, len(recommendations_df) + 1)
    )

    return recommendations_df

### 7.1 Evaluate a Single Validation Example

Before evaluating the complete validation dataset, one interaction is inspected to confirm that the ranking and relevance comparison work correctly.

In [25]:
validation_example = validation_df.iloc[0]

validation_task_id = validation_example["task_id"]
actual_agent_id = validation_example["agent_id"]

validation_recommendations = recommend_agents(
    validation_task_id,
    top_k=5,
    enforce_availability=False
)

print("Task:", validation_task_id)
print("Historical Agent:", actual_agent_id)

validation_recommendations

Task: T1892
Historical Agent: A058


,rank,agent_id,score,skill_match,specialization_match,location_match,performance_score,workload_score,experience_score
0,1,A042,0.8788,1.0,1.0,0.0,0.9875,0.8,1.000
1,2,A026,0.8731,1.0,1.0,1.0,0.9063,0.2,0.250
2,3,A005,0.8595,1.0,1.0,0.0,0.9829,0.8,0.625
3,4,A055,0.8512,1.0,1.0,0.0,0.6369,1.0,0.750
4,5,A070,0.8442,1.0,1.0,0.0,0.7541,1.0,0.375


In [26]:
recommended_agent_ids = (
    validation_recommendations[
        "agent_id"
    ].tolist()
)

print(
    "Historical agent found in Top 5:",
    actual_agent_id in recommended_agent_ids
)

Historical agent found in Top 5: False


### 7.2 Define Ranking Evaluation Metrics

The baseline is evaluated using ranking metrics that are appropriate for recommender systems.

Hit Rate@K measures whether the historically assigned agent appears anywhere within the Top-K recommendations.

Recall@K measures whether the relevant agent was retrieved.

Precision@K measures the proportion of the Top-K recommendations represented by the observed relevant agent.

Average Precision rewards the recommender when the relevant agent appears closer to the top of the ranked list.

In [27]:
def evaluate_single_task(
    task_id,
    relevant_agent_id,
    k=5
):

    recommendations = recommend_agents(
        task_id,
        top_k=k,
        enforce_availability=False
    )

    recommended_agents = (
        recommendations["agent_id"].tolist()
    )

    if relevant_agent_id in recommended_agents:

        rank = (
            recommended_agents.index(
                relevant_agent_id
            ) + 1
        )

        hit = 1
        recall = 1.0
        precision = 1.0 / k
        average_precision = 1.0 / rank

    else:

        rank = None
        hit = 0
        recall = 0.0
        precision = 0.0
        average_precision = 0.0

    return {
        "task_id": task_id,
        "relevant_agent_id": relevant_agent_id,
        "hit": hit,
        "precision": precision,
        "recall": recall,
        "average_precision": average_precision,
        "relevant_agent_rank": rank
    }

### 7.3 Evaluate the Full Validation Dataset

The baseline recommender is evaluated across all validation interactions.

The test dataset remains untouched during this stage.

In [28]:
def evaluate_validation_set(
    validation_data,
    k=5
):

    results = []

    for _, row in validation_data.iterrows():

        result = evaluate_single_task(
            task_id=row["task_id"],
            relevant_agent_id=row["agent_id"],
            k=k
        )

        results.append(result)

    return pd.DataFrame(results)

In [29]:
validation_results_k5 = (
    evaluate_validation_set(
        validation_df,
        k=5
    )
)

validation_results_k5.head()

,task_id,relevant_agent_id,hit,precision,recall,average_precision,relevant_agent_rank
0,T1892,A058,0,0.0,0.0,0.0,NaN
1,T1981,A030,0,0.0,0.0,0.0,NaN
2,T1079,A059,0,0.0,0.0,0.0,NaN
3,T517,A060,0,0.0,0.0,0.0,NaN
4,T1923,A044,0,0.0,0.0,0.0,NaN


In [30]:
precision_at_5 = (
    validation_results_k5[
        "precision"
    ].mean()
)

recall_at_5 = (
    validation_results_k5[
        "recall"
    ].mean()
)

hit_rate_at_5 = (
    validation_results_k5[
        "hit"
    ].mean()
)

map_at_5 = (
    validation_results_k5[
        "average_precision"
    ].mean()
)

print(
    f"Precision@5: {precision_at_5:.4f}"
)

print(
    f"Recall@5: {recall_at_5:.4f}"
)

print(
    f"Hit Rate@5: {hit_rate_at_5:.4f}"
)

print(
    f"MAP@5: {map_at_5:.4f}"
)

Precision@5: 0.0510
Recall@5: 0.2551
Hit Rate@5: 0.2551
MAP@5: 0.1496


In [31]:
evaluation_summary = []

for k in [1, 3, 5]:

    results_k = evaluate_validation_set(
        validation_df,
        k=k
    )

    evaluation_summary.append({
        "K": k,
        "Precision@K":
            results_k["precision"].mean(),

        "Recall@K":
            results_k["recall"].mean(),

        "Hit Rate@K":
            results_k["hit"].mean(),

        "MAP@K":
            results_k[
                "average_precision"
            ].mean()
    })


evaluation_summary_df = pd.DataFrame(
    evaluation_summary
)

evaluation_summary_df

,K,Precision@K,Recall@K,Hit Rate@K,MAP@K
0,1,0.091837,0.091837,0.091837,0.091837
1,3,0.069728,0.209184,0.209184,0.138605
2,5,0.051020,0.255102,0.255102,0.149575


### Baseline Validation Results

The content-based baseline achieved a Hit Rate@1 of approximately 9.18%, Hit Rate@3 of 20.92% and Hit Rate@5 of 25.51%.

This means that the historically assigned agent appeared as the first recommendation for approximately 9% of validation tasks and appeared within the top five recommendations for approximately 26% of validation tasks.

MAP also increased from approximately 9.18% at K=1 to 14.96% at K=5, indicating that additional relevant historical agents were retrieved as the recommendation list was expanded.

The baseline results provide an initial reference point rather than a final performance target. The project will later compare this content-based approach with collaborative filtering, knowledge graph reasoning and the final hybrid recommender.

Because the current dataset is synthetic and contains only one observed assigned agent for each historical task, these results are treated as development evidence rather than final real-world model performance.

### Interpretation

The validation results provide the initial performance evidence for the content-based baseline.

Hit Rate and Recall indicate how often the historically assigned agent appeared within the recommended agents, while MAP also considers the position of that agent in the ranking.

These results represent development evidence using synthetic data and are not treated as final real-world performance.

The validation metrics will later be used to compare the content-based baseline with collaborative filtering, knowledge graph and hybrid recommendation approaches.

The test dataset has not been used during this development stage and remains reserved for final evaluation.

## 8. Compare Against a Random Baseline

A simple random baseline is used as a reference point for evaluating the content-based recommender.

The random baseline selects agents without considering skills, specialization, location, workload, performance or experience.

The purpose of this comparison is to determine whether the content-based recommender provides better rankings than random agent selection.

In [32]:
random_baseline_results = pd.DataFrame({
    "K": [1, 3, 5],
    "Random Hit Rate": [
        1 / len(agents_df),
        3 / len(agents_df),
        5 / len(agents_df)
    ]
})

random_baseline_results

,K,Random Hit Rate
0,1,0.01
1,3,0.03
2,5,0.05


In [33]:
baseline_comparison_df = (
    evaluation_summary_df[
        ["K", "Hit Rate@K"]
    ]
    .merge(
        random_baseline_results,
        on="K"
    )
)

baseline_comparison_df[
    "Improvement Factor"
] = (
    baseline_comparison_df["Hit Rate@K"]
    / baseline_comparison_df["Random Hit Rate"]
)

baseline_comparison_df

,K,Hit Rate@K,Random Hit Rate,Improvement Factor
0,1,0.091837,0.01,9.183673
1,3,0.209184,0.03,6.972789
2,5,0.255102,0.05,5.102041


### Interpretation of Baseline Comparison

The content-based recommender performed better than random agent selection at all evaluated recommendation depths.

At Top-1, the model achieved a Hit Rate of approximately 9.18% compared with 1% for random selection, representing an improvement of approximately 9.2 times.

At Top-3, the model achieved approximately 20.92% compared with 3% for random selection.

At Top-5, the model achieved approximately 25.51% compared with 5% for random selection, representing an improvement of approximately 5.1 times.

These results show that the content-based features provide useful information for ranking agents. However, because the experiment currently uses synthetic development data, the results are treated as baseline development evidence rather than final real-world performance.

## 9. Save Baseline Experiment Evidence

The configuration, validation metrics and prediction results from the baseline experiment are saved so that the experiment can be reproduced and compared with later recommendation approaches.

The evidence generated from this experiment is development evidence based on synthetic data.

In [34]:
import json

experiment_path = (
    project_root
    / "experiments"
    / "CBF_RUN_001"
)

experiment_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Experiment folder:", experiment_path)

Experiment folder: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation\experiments\CBF_RUN_001


In [35]:
experiment_config = {
    "run_id": "CBF_RUN_001",
    "model": "Content-Based Baseline",
    "dataset_version": "synthetic_v1",
    "random_seed": 42,
    "training_split": 0.70,
    "validation_split": 0.15,
    "test_split": 0.15,
    "evaluation_data": "validation",
    "test_data_used": False,
    "weights": BASELINE_WEIGHTS
}

with open(
    experiment_path / "config.json",
    "w"
) as file:
    json.dump(
        experiment_config,
        file,
        indent=4
    )

print("Configuration saved.")

Configuration saved.


In [36]:
evaluation_summary_df.to_csv(
    experiment_path
    / "validation_metrics.csv",
    index=False
)

baseline_comparison_df.to_csv(
    experiment_path
    / "random_baseline_comparison.csv",
    index=False
)

print("Evaluation metrics saved.")

Evaluation metrics saved.


In [37]:
validation_results_k5.to_csv(
    experiment_path
    / "validation_predictions_k5.csv",
    index=False
)

print("Validation predictions saved.")

Validation predictions saved.


In [38]:
for file in experiment_path.iterdir():
    print(file.name)

config.json
random_baseline_comparison.csv
validation_metrics.csv
validation_predictions_k5.csv


## Conclusion

The first content-based recommendation baseline was successfully developed and evaluated.

The recommender ranks agents using skill compatibility, specialization, location, workload, performance and experience, while also generating human-readable explanations for its recommendations.

On the synthetic validation dataset, the model achieved a Hit Rate@5 of approximately 25.51%, compared with approximately 5% for random agent selection.

This provides the baseline against which collaborative filtering, knowledge graph reasoning and the final hybrid recommender will be compared.

The test dataset remains unused and reserved for later final evaluation.

## 10. Minimum Suitability Threshold

The recommender should not force a recommendation when no agent is sufficiently suitable for a task.

A minimum suitability threshold is therefore used. If the highest-ranked available agent has a score below the threshold, the task is flagged for supervisor review instead of being automatically recommended for assignment.

The threshold used during development is 0.60. This value is an initial operational rule and may be adjusted later using validation evidence and supervisor feedback.